In [1]:
import torch
from torch import nn
from torch.utils import data
import torchvision
from torchvision import transforms

trans = transforms.ToTensor()
train_data = torchvision.datasets.FashionMNIST(root = '../data', train = True, transform = trans, 
                                               download = False)
test_data = torchvision.datasets.FashionMNIST(root = '../data', train = False, transform = trans, 
                                               download = False)
batch_size = 256
train_iter = data.DataLoader(train_data, batch_size = batch_size, shuffle = True)
test_iter = data.DataLoader(test_data, batch_size = batch_size, shuffle = False)

In [2]:
class Reshape(nn.Module):
    def forward(self, X): # forward要有self实例参数
        return X.reshape(-1, 1, 28, 28) # 对输入增加c_in通道 黑白图片 channel = 1
net = nn.Sequential(
    Reshape(),
    # 两层完整卷积层
    nn.Conv2d(1, 6, kernel_size = 5, padding = 2), # out: 6@28×28
    nn.Sigmoid(), # 激活函数
    nn.AvgPool2d(2), # 平均池化层 取半 6@14×14
    nn.Conv2d(6, 16, kernel_size = 5), # out: 16@10×10
    nn.Sigmoid(),
    nn.AvgPool2d(2), # out: 16@5×5
    # 多感知机
    nn.Flatten(), # 16×5×5
    nn.Linear(16*5*5, 120),
    nn.Sigmoid(),
    nn.Linear(120, 84),
    nn.Sigmoid(),
    nn.Linear(84, 10)
)

In [3]:
def weight_init(layer):
    if type(layer) == nn.Linear or type(layer) == nn.Conv2d:
        nn.init.xavier_uniform_(layer.weight) # 避免梯度消失或爆炸
net.apply(weight_init)

Sequential(
  (0): Reshape()
  (1): Conv2d(1, 6, kernel_size=(5, 5), stride=(1, 1), padding=(2, 2))
  (2): Sigmoid()
  (3): AvgPool2d(kernel_size=2, stride=2, padding=0)
  (4): Conv2d(6, 16, kernel_size=(5, 5), stride=(1, 1))
  (5): Sigmoid()
  (6): AvgPool2d(kernel_size=2, stride=2, padding=0)
  (7): Flatten(start_dim=1, end_dim=-1)
  (8): Linear(in_features=400, out_features=120, bias=True)
  (9): Sigmoid()
  (10): Linear(in_features=120, out_features=84, bias=True)
  (11): Sigmoid()
  (12): Linear(in_features=84, out_features=10, bias=True)
)

In [4]:
loss = nn.CrossEntropyLoss()

In [5]:
updater = torch.optim.SGD(net.parameters(), lr = 0.9)

In [6]:
def accuracy(y_hat, y):
    y_hat = torch.argmax(y_hat, dim = 1)
    return (y_hat.type(y.dtype) == y).sum()

In [7]:
def evaluate_acc(net, test_iter):
    if isinstance(net, nn.Module):
        net.eval()
    sum = 0
    num = 0
    for X, y in test_iter:
        y_hat = net(X)
        sum += accuracy(y_hat, y)
        num += y.numel()
    return (sum / num).item()

In [8]:
def train(train_iter, test_iter, net, loss, updater, num_epochs):
    for epoch in range(num_epochs):
        train_loss = 0
        acc_num = 0
        num_example = 0
        net.train()
        for X, y in train_iter:
            y_hat = net(X)
            l = loss(y_hat, y)
            updater.zero_grad()
            l.backward()
            updater.step()
            train_loss += (l.item())*(y.numel())
            num_example += y.numel()
            acc_num += accuracy(y_hat, y)
        test_acc = evaluate_acc(net, test_iter)
        print(f'epoch {epoch +1}, loss {train_loss / num_example}, train_acc {acc_num / num_example}, test_acc {test_acc}')
num_epochs = 30
train(train_iter, test_iter, net, loss, updater, num_epochs)

epoch 1, loss 2.3204202058156334, train_acc 0.10041666775941849, test_acc 0.10000000149011612
epoch 2, loss 1.9209915247599283, train_acc 0.25591665506362915, test_acc 0.5690000057220459
epoch 3, loss 0.9410152491251628, train_acc 0.6267833113670349, test_acc 0.6100999712944031
epoch 4, loss 0.7467785062472025, train_acc 0.7089499831199646, test_acc 0.7208999991416931
epoch 5, loss 0.6525535102844239, train_acc 0.745199978351593, test_acc 0.7023000121116638
epoch 6, loss 0.5956473980903626, train_acc 0.7699499726295471, test_acc 0.767799973487854
epoch 7, loss 0.5561300567309062, train_acc 0.7856333255767822, test_acc 0.7803000211715698
epoch 8, loss 0.5205918427149455, train_acc 0.8006166815757751, test_acc 0.8007000088691711
epoch 9, loss 0.489801523621877, train_acc 0.8152333498001099, test_acc 0.8098000288009644
epoch 10, loss 0.4721517058054606, train_acc 0.8241166472434998, test_acc 0.8072999715805054
epoch 11, loss 0.45401244633992516, train_acc 0.8314999938011169, test_acc 0.80